In [ ]:
import hashlib
import secrets

# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")


# Module 7: Self-Assessment & Exercises

## 7.1 Concept Check

Answer each question, then run the verification cell below.

---

In [24]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 1: Finite Field Arithmetic
# ═══════════════════════════════════════════════════════════════
#
#  Compute the multiplicative inverse of 7 in F_11.
#  That is, find x such that 7 * x ≡ 1 (mod 11).

answer_1 = 0  # <-- Replace with your answer

# Verify
if (7 * answer_1) % 11 == 1:
    print(f"Exercise 1: ✓  7 × {answer_1} = {7 * answer_1} ≡ {(7*answer_1)%11} (mod 11)")
else:
    print(f"Exercise 1: ✗  7 × {answer_1} = {7 * answer_1} ≡ {(7*answer_1)%11} (mod 11), need ≡ 1")
    print(f"  Hint: use Fermat's little theorem: 7^(11-2) mod 11")

Exercise 1: ✗  7 × 0 = 0 ≡ 0 (mod 11), need ≡ 1
  Hint: use Fermat's little theorem: 7^(11-2) mod 11


In [25]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 2: Points on a Curve
# ═══════════════════════════════════════════════════════════════
#
#  For the curve y² = x³ + 7 over F_11, find a valid point.
#  Try x values and check if x³ + 7 is a quadratic residue mod 11.
#  If it is, compute y.

x_answer = 0  # <-- Replace with x
y_answer = 0  # <-- Replace with y

# Verify
p_ex = 11
lhs_ex = (y_answer ** 2) % p_ex
rhs_ex = (x_answer ** 3 + 7) % p_ex
if lhs_ex == rhs_ex and not (x_answer == 0 and y_answer == 0):
    print(f"Exercise 2: ✓  ({x_answer}, {y_answer}) is on y² = x³ + 7 over F_11")
    print(f"  y² = {y_answer}² = {y_answer**2} ≡ {lhs_ex} (mod 11)")
    print(f"  x³+7 = {x_answer}³+7 = {x_answer**3+7} ≡ {rhs_ex} (mod 11)")
else:
    print(f"Exercise 2: ✗  ({x_answer}, {y_answer}) is NOT on the curve")
    print(f"  y² mod 11 = {lhs_ex}, x³+7 mod 11 = {rhs_ex}")
    print(f"  Hint: try x=2. What is 2³ + 7 = {2**3 + 7} mod 11 = {(2**3 + 7) % 11}?")
    print(f"  Is {(2**3 + 7) % 11} a perfect square mod 11?")

Exercise 2: ✗  (0, 0) is NOT on the curve
  y² mod 11 = 0, x³+7 mod 11 = 7
  Hint: try x=2. What is 2³ + 7 = 15 mod 11 = 4?
  Is 4 a perfect square mod 11?


In [26]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 3: Point Negation
# ═══════════════════════════════════════════════════════════════
#
#  If P = (3, 10) on curve y² = x³ + 7 over F_11,
#  what is -P?

neg_x = 0  # <-- Replace
neg_y = 0  # <-- Replace

# Verify
expected_neg_y = (11 - 10) % 11
if neg_x == 3 and neg_y == expected_neg_y:
    print(f"Exercise 3: ✓  -P = ({neg_x}, {neg_y})")
    print(f"  Negation flips y: -(3, 10) = (3, 11-10) = (3, {expected_neg_y})")
else:
    print(f"Exercise 3: ✗  Got ({neg_x}, {neg_y})")
    print(f"  Hint: negation = (x, P-y) = (3, 11-10) = ?")

Exercise 3: ✗  Got (0, 0)
  Hint: negation = (x, P-y) = (3, 11-10) = ?


In [27]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 4: Double-and-Add Trace
# ═══════════════════════════════════════════════════════════════
#
#  How many point doublings and point additions does it take
#  to compute 45 × G using double-and-add?
#
#  Steps:
#  1. Write 45 in binary: ____________
#  2. Count the bits (= number of doublings)
#  3. Count the 1-bits (= number of additions)

binary_45 = ""     # <-- Write 45 in binary (e.g., "110")
num_doubles = 0     # <-- How many doublings?
num_adds = 0        # <-- How many additions?

# Verify
actual_binary = bin(45)[2:]
actual_doubles = len(actual_binary) - 1  # Don't double on last bit
actual_adds = actual_binary.count('1')

if binary_45 == actual_binary:
    print(f"Exercise 4a: ✓  45 in binary = {actual_binary}")
else:
    print(f"Exercise 4a: ✗  45 in binary = {actual_binary}, you wrote '{binary_45}'")

if num_doubles == actual_doubles:
    print(f"Exercise 4b: ✓  {actual_doubles} doublings")
else:
    print(f"Exercise 4b: ✗  Expected {actual_doubles} doublings, got {num_doubles}")

if num_adds == actual_adds:
    print(f"Exercise 4c: ✓  {actual_adds} additions")
else:
    print(f"Exercise 4c: ✗  Expected {actual_adds} additions, got {num_adds}")

print(f"\nNaive would need 44 additions. Double-and-add: {actual_doubles + actual_adds} operations.")

Exercise 4a: ✗  45 in binary = 101101, you wrote ''
Exercise 4b: ✗  Expected 5 doublings, got 0
Exercise 4c: ✗  Expected 4 additions, got 0

Naive would need 44 additions. Double-and-add: 9 operations.


In [28]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 5: ECDSA — Verify by Hand
# ═══════════════════════════════════════════════════════════════
#
#  Given the ECDSA verification equation:
#    R' = u1×G + u2×P
#  where u1 = z/s mod N and u2 = r/s mod N
#
#  Fill in the proof that R' = R (the original nonce point).
#  
#  Starting from: R' = (z/s)×G + (r/s)×P
#  Substitute P = d×G:
#    R' = (z/s)×G + (r/s)×(d×G)
#       = _____________ × G        ← factor out G
#  Since s = k⁻¹(z + r×d):
#    (z + r×d)/s = (z + r×d) / (k⁻¹(z + r×d)) = _____
#  Therefore R' = _____ × G = R    QED

factored = ""       # <-- What's the scalar before ×G? (e.g., "(z+rd)/s")
simplified = ""     # <-- What does (z+rd)/s simplify to?

correct_factored = "(z+rd)/s"
correct_simplified = "k"

if factored.replace(" ", "") == correct_factored:
    print(f"Exercise 5a: ✓  R' = {factored} × G")
else:
    print(f"Exercise 5a: ✗  R' = {correct_factored} × G")

if simplified.lower().strip() == correct_simplified:
    print(f"Exercise 5b: ✓  Simplifies to {simplified}, so R' = k×G = R  ✓")
else:
    print(f"Exercise 5b: ✗  (z+rd)/s simplifies to k (because s = (z+rd)/k)")

Exercise 5a: ✗  R' = (z+rd)/s × G
Exercise 5b: ✗  (z+rd)/s simplifies to k (because s = (z+rd)/k)


In [29]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 6: Implement Nonce Recovery
# ═══════════════════════════════════════════════════════════════
#
#  Given two signatures (r, s1) and (r, s2) for messages with
#  hashes z1 and z2 (same r means same nonce!), recover the
#  nonce k and then the private key d.
#
#  Formulas:
#    k = (z1 - z2) × (s1 - s2)⁻¹ mod N
#    d = r⁻¹ × (s1×k - z1) mod N

# Setup (don't modify)
ex6_d = secrets.randbelow(SECP_N - 1) + 1
ex6_k = secrets.randbelow(SECP_N - 1) + 1
ex6_R = scalar_mult(ex6_k, G)
ex6_r = ex6_R.x % SECP_N
ex6_k_inv = pow(ex6_k, SECP_N - 2, SECP_N)

ex6_z1 = int.from_bytes(hashlib.sha256(b"message one").digest(), 'big')
ex6_z2 = int.from_bytes(hashlib.sha256(b"message two").digest(), 'big')
ex6_s1 = (ex6_k_inv * (ex6_z1 + ex6_r * ex6_d)) % SECP_N
ex6_s2 = (ex6_k_inv * (ex6_z2 + ex6_r * ex6_d)) % SECP_N

# YOUR CODE: recover k and d
recovered_k = 0  # <-- Replace with formula
recovered_d = 0  # <-- Replace with formula

# Verify
if recovered_k == ex6_k:
    print(f"Exercise 6a: ✓  Nonce k recovered!")
else:
    print(f"Exercise 6a: ✗  k not recovered")
    print(f"  Hint: k = (z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N) % SECP_N")

if recovered_d == ex6_d:
    print(f"Exercise 6b: ✓  Private key d recovered! (This is why nonce reuse is fatal)")
else:
    print(f"Exercise 6b: ✗  d not recovered")
    print(f"  Hint: d = pow(r, SECP_N - 2, SECP_N) * (s1 * k - z1) % SECP_N")

Exercise 6a: ✗  k not recovered
  Hint: k = (z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N) % SECP_N
Exercise 6b: ✗  d not recovered
  Hint: d = pow(r, SECP_N - 2, SECP_N) * (s1 * k - z1) % SECP_N


## 7.2 Knowledge Map

Check off each concept as you understand it:

```
Module 1: Foundations                    Module 4: ElGamal vs ECC
[ ] Abelian group axioms                [ ] Discrete log in both settings
[ ] Finite field F_p                    [ ] Key size comparison
[ ] Modular inverse via Fermat          [ ] ECDH key exchange
[ ] Equivalence classes mod n           [ ] ECC encryption/decryption
[ ] Projective coordinates
                                        Module 5: ECDSA
Module 2: Elliptic Curves               [ ] Signature generation
[ ] Weierstrass equation                [ ] Signature verification
[ ] secp256k1 parameters                [ ] The nonce catastrophe
[ ] Curve over finite field             [ ] Recoverable signatures
[ ] Non-singularity condition
                                        Module 6: Bitcoin
Module 3: Point Operations              [ ] Schnorr vs ECDSA
[ ] Point addition formula              [ ] ECDH in onion routing
[ ] Point doubling formula              [ ] Ephemeral key blinding
[ ] Point at infinity (identity)
[ ] Scalar multiplication
[ ] Double-and-add algorithm
[ ] Compressed public keys
```

---

## 7.3 Further Study

### External references:

- [SEC 2: Recommended Elliptic Curve Domain Parameters](https://www.secg.org/sec2-v2.pdf) (secp256k1 spec)
- [BIP340: Schnorr Signatures](https://github.com/bitcoin/bips/blob/master/bip-0340.mediawiki)
- [BOLT #4: Onion Routing Protocol](https://github.com/lightning/bolts/blob/master/04-onion-routing.md)
- Johnson, Menezes, Vanstone: *The Elliptic Curve Digital Signature Algorithm (ECDSA)*, Certicom 2001

### Security considerations:

All implementations in this notebook are **educational**. Production code must:
- Use constant-time operations (timing attacks)
- Use battle-tested libraries (`libsecp256k1`, `ring`, `openssl`)
- Validate all inputs (point-on-curve, subgroup order)
- Never reuse nonces
- Zeroize secrets after use
